# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
url = croissant_url

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset overview
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Dataset @id: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect which record sets are available and list their `@id`s and the fields within each record set.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.recordSet
record_sets_list = []
fields_per_recordset = {}
print("Available Record Sets:")
for rs in record_sets:
    rs_id = rs.id
    rs_name = getattr(rs, 'name', '')
    print(f"- RecordSet name: {rs_name}, @id: {rs_id}")
    record_sets_list.append(rs_id)
    fields = getattr(rs, 'field', [])
    print("  Fields in RecordSet:")
    field_ids = []
    for f in fields:
        field_label = getattr(f, 'label', getattr(f, 'name', ''))
        print(f"    - Field: {field_label}, @id: {f.id}, dataType: {getattr(f, 'dataType', '')}")
        field_ids.append(f.id)
    fields_per_recordset[rs_id] = field_ids
if not record_sets_list:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will load each record set. If there are multiple record sets, we create a DataFrame for each.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets_list:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records from record set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {str(e)}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (for example, `score_gad7`, `score_phq9`, or `score_psq` if present), filter for records with high scores, normalize the scores, and group by a demographic field such as `gender` or `level_of_education`.

In [ ]:
# Example EDA: Filter, Normalize, Group
# Choose a record set (first in list)
if len(record_sets_list) > 0:
    record_set_id = record_sets_list[0]
    df = dataframes[record_set_id]
    # Determine numeric fields in the record set
    candidate_fields = ['score_gad7', 'score_phq9', 'score_psq']
    numeric_field = None
    for f in candidate_fields:
        if f in df.columns:
            numeric_field = f
            break
    if numeric_field:
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Choose a group field
        potential_group_fields = ['gender', 'level_of_education', 'marital_status', 'village']
        group_field = None
        for gf in potential_group_fields:
            if gf in filtered_df.columns:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No group field found in filtered DataFrame.")
    else:
        print("No recognized numeric field for EDA.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll show a histogram of a selected numeric score and, if grouped, a bar plot by a demographic field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distributions
if len(record_sets_list) > 0 and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we've loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- The dataset includes demographic and mental health survey data.
- We reviewed record sets and fields, extracted the data, filtered based on numeric scores, and visualized distributions.
- Higher scores on the GAD-7, PHQ-9, and PSQ are observable and can be analyzed in the context of demographic variables.
- This dataset can support deeper research into mental health trends and interventions in the Kenyan context.

For more advanced analysis, consider exploring relationships using statistical tests or modeling approaches, and consult the Croissant schema for additional metadata or record set details.